# REINFORCE — solve CartPole, a real RL benchmark

**Agents & RL** · the simplest policy-gradient method (the root of PPO, used in RLHF).

We train on **Gymnasium `CartPole-v1`** — the classic control benchmark: balance a pole on a cart by pushing left/right. Reward every step the pole stays up, and push up the probability of actions that led to high return, using a **value baseline** (actor-critic) + an entropy bonus. "Solved" means an episode return near the maximum of **500**.

> CPU is fine — trains in a couple of minutes.

In [ ]:
!pip -q install gymnasium

In [ ]:
import os, torch, torch.nn as nn, torch.nn.functional as F, matplotlib.pyplot as plt
from torch.distributions import Categorical
import gymnasium as gym
device = "cuda" if torch.cuda.is_available() else "cpu"
EPISODES = int(os.environ.get("EPISODES", 600)); GAMMA = 0.99; torch.manual_seed(0)
env = gym.make("CartPole-v1")
print("obs", env.observation_space.shape, "| actions", env.action_space.n)

## 1 · Actor (policy) + critic (value baseline)

In [ ]:
policy = nn.Sequential(nn.Linear(4, 64), nn.Tanh(), nn.Linear(64, 2)).to(device)   # actor: state -> action logits
critic = nn.Sequential(nn.Linear(4, 64), nn.Tanh(), nn.Linear(64, 1)).to(device)   # value baseline
opt = torch.optim.Adam(list(policy.parameters()) + list(critic.parameters()), 3e-3)

## 2 · Roll out one episode under the current policy

In [ ]:
def rollout(seed=None):
    s, _ = env.reset(seed=seed); logps, vals, ents, rews = [], [], [], []
    done = False
    while not done:
        st = torch.tensor(s, dtype=torch.float32, device=device); d = Categorical(logits=policy(st)); a = d.sample()
        logps.append(d.log_prob(a)); ents.append(d.entropy()); vals.append(critic(st).squeeze())
        s, r, term, trunc, _ = env.step(int(a)); done = term or trunc; rews.append(r)
    return logps, vals, ents, rews

## 3 · Train — REINFORCE with a value baseline + entropy bonus

In [ ]:
hist = []; ep_returns = []
for ep in range(EPISODES):
    logps, vals, ents, rews = rollout(seed=ep)
    G = 0; returns = []
    for r in reversed(rews): G = r + GAMMA * G; returns.insert(0, G)
    returns = torch.tensor(returns, device=device); vals = torch.stack(vals)
    adv = (returns - vals.detach()); adv = (adv - adv.mean()) / (adv.std(unbiased=False) + 1e-8)
    loss = -(torch.stack(logps) * adv).mean() - 0.01 * torch.stack(ents).mean() + 0.5 * F.mse_loss(vals, returns)
    opt.zero_grad(); loss.backward(); opt.step()
    ep_returns.append(sum(rews))
    if ep % max(1, EPISODES // 12) == 0:
        ma = sum(ep_returns[-20:]) / len(ep_returns[-20:]); hist.append((ep, round(ma, 1))); print(f"ep {ep:4d}  avg return {ma:6.1f}")

## 4 · Evaluate the greedy policy + plot the learning curve

In [ ]:
evals = []
for k in range(20):
    s, _ = env.reset(seed=1000 + k); done = False; tot = 0
    while not done:
        with torch.no_grad(): a = int(policy(torch.tensor(s, dtype=torch.float32, device=device)).argmax())
        s, r, term, trunc, _ = env.step(a); done = term or trunc; tot += r
    evals.append(tot)
greedy_eval = sum(evals) / len(evals); print(f"greedy eval mean return {greedy_eval:.1f}  (max 500)")
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(*zip(*hist), "-o"); ax.axhline(500, ls="--", c="C7", label="max (500)")
ax.set_xlabel("episode"); ax.set_ylabel("avg return (20-ep window)"); ax.grid(alpha=.3); ax.legend(); ax.set_title("REINFORCE solves CartPole")
plt.show()

## Save artifacts — checkpoint · metrics · figure
Everything is written to **outputs/AG_reinforce_gridworld/** — the model checkpoint, the full loss/eval history as JSON, and the final figure — then zipped. Colab sessions are ephemeral, so the last lines show how to download the zip or copy it to Google Drive.

In [ ]:
import os, json, torch, shutil
run = "outputs/AG_reinforce_gridworld"; os.makedirs(run, exist_ok=True)
torch.save(policy.state_dict(), f"{run}/policy.pt")
json.dump({"return": hist, "greedy_eval": greedy_eval}, open(f"{run}/metrics.json", "w"), indent=2)
try:
    fig.savefig(f"{run}/figure.png", dpi=120, bbox_inches="tight")
except Exception: pass
shutil.make_archive(run, "zip", run)
print("saved to", run, "->", sorted(os.listdir(run)))
# keep it past the session:  from google.colab import files; files.download(f"{run}.zip")
# or mount Drive:  from google.colab import drive; drive.mount('/content/drive')  # then shutil.copytree(run, "/content/drive/MyDrive/"+run)

## (Optional) Publish to the Hugging Face Hub
Upload this run as a **model repo** — the checkpoint, `metrics.json` (full loss/eval history) and the results figure, embedded in an auto-generated model card. Do it for each lab, then group them into a **Collection** on your HF profile (Profile → New collection), or with the commented `add_collection_item` call below. It needs a **write token**, so it only runs once you sign in.

In [ ]:
# (optional) publish this run as a Hugging Face model repo — checkpoint + metrics + plot
!pip -q install huggingface_hub
from huggingface_hub import HfApi, notebook_login
import os
notebook_login()   # paste a WRITE token from https://huggingface.co/settings/tokens
api = HfApi(); user = api.whoami()["name"]
lab = os.path.basename(run); repo_id = f"{user}/" + lab.lower().replace("_", "-")
fig = "\n![results](figure.png)\n" if os.path.exists(f"{run}/figure.png") else ""
open(f"{run}/README.md", "w").write("---\ntags: [ropedia-academy, education]\n---\n# " + lab + "\n\nTrained in **Ropedia Academy** (educational lab). Checkpoint, full loss/eval history (metrics.json) and the results figure are included." + fig)
api.create_repo(repo_id, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=run, repo_id=repo_id, commit_message="Upload from Ropedia Academy")
print("uploaded ->", "https://huggingface.co/" + repo_id)
# group everything into one Collection (run once, after a few uploads):
# from huggingface_hub import create_collection, add_collection_item
# col = create_collection("Ropedia Academy - trained models", namespace=user, exists_ok=True)
# add_collection_item(col.slug, item_id=repo_id, item_type="model")

## How this links to tracks A–D
Policy gradients scale up to **D · Scene / world** embodied control.

### Where to go next
- Add clipped updates → **PPO** (the algorithm behind RLHF), or a replay buffer → off-policy methods.
- Swap CartPole for a harder Gym/MuJoCo task; learn from pixels with a world model (track D).